In [2]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../")

from ease_recommender import *
from npmi_recommender import *

import pickle as p

def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

loading cache data...
building csr matrices...
done
a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412
c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71
d_name='Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb)'
Num Rows: 12615
e_name='Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)'
Num Rows: 32356


In [3]:
mat = artist_mat
mat = csr_array(mat)

n_users, n_items = mat.shape

X = mat.T @ mat
# X = X / n_users

X.shape

(295860, 295860)

In [25]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.tocoo().nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [26]:
alpha = .1 * .85

# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True)
# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=False)
# sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True, normalize=True)

sppmi = sparse_laplace_sppmi(X, alpha=alpha, zero_diag=False)

metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

11.0

In [27]:
def get_metric(i, j, sppmi):
    return np.argsort(-sppmi[i].toarray()).tolist().index(j)

metrics = [
    get_metric(a, b, sppmi),
    get_metric(b, a, sppmi),
    
#     get_metric(b, c, sppmi),
#     get_metric(c, b, sppmi),
]

np.mean(metrics)

np.float64(11.0)

In [173]:
import numpy as np
import scipy.sparse as sp
import rustworkx as rx
import time

class GraphAnalyzer:
    def __init__(self, csr_matrix, threshold=0.0):
        """
        Initializes the graph from a sparse CSR matrix.
        
        Args:
            csr_matrix: Scipy CSR array of unnormalized similarity scores.
            threshold: Minimum score to keep an edge. Crucial for performance 
                       if the similarity matrix is dense.
        """
        self.n_nodes = csr_matrix.shape[0]
        
        # 1. Efficient Conversion: CSR -> Rustworkx
        # We convert to COO format first to extract (row, col, weight) tuples efficiently
        # without iterating in pure Python.
        coo = csr_matrix.tocoo()
        
        # Apply threshold filtering
        if threshold > 0:
            mask = coo.data > threshold
            rows = coo.row[mask]
            cols = coo.col[mask]
            weights = coo.data[mask]
        else:
            rows = coo.row
            cols = coo.col
            weights = coo.data

        # 2. Build the graph
        # We use PyDiGraph (Directed) because RWR relies on directionality.
        # If your similarities are symmetric, edges (u,v) and (v,u) will both be added.
        self.graph = rx.PyDiGraph()
        self.graph.add_nodes_from(range(self.n_nodes))
        
        # Zip into edge tuples: (source, target, weight)
        # We cast weights to float to ensure Rust handles them correctly
        edge_list = list(zip(rows, cols, weights.astype(float)))
        
        # Bulk add is significantly faster than adding one by one
        self.graph.add_edges_from(edge_list)
        
        print(f"✅ Graph initialized with {self.graph.num_nodes()} nodes and {self.graph.num_edges()} edges.")

    def random_walk_with_restart(self, seed_node, restart_prob=0.15):
        """
        Performs Random Walk with Restarts (RWR) from a specific seed node.
        
        RWR is mathematically equivalent to Personalized PageRank where the 
        personalization vector is concentrated entirely on the seed node.
        """
        # Damping factor alpha = 1 - restart_probability
        alpha = 1.0 - restart_prob
        
        # Personalization: {node_index: score}
        # We focus 100% of the personalization on the seed node
        personalization = {seed_node: 1.0}
        
        # rustworkx.pagerank automatically handles weight normalization 
        # (converting similarity scores to transition probabilities)
        scores = rx.pagerank(
            self.graph, 
            alpha=alpha, 
            weight_fn=lambda edge: edge, # Function to extract weight from edge payload
            personalization=personalization,
            tol=1e-6
        )
        
        # Convert dictionary {node: score} to a sorted list or array
        # Returning top 10 for immediate utility
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_scores

    def betweenness_centrality(self, k=None, normalized=True):
        """
        Computes Betweenness Centrality.
        
        Args:
            k: (int) If provided, approximates centrality using k pivot nodes.
               Much faster for large graphs.
        """
        # This function runs in parallel threads in the Rust backend
        bc_scores = rx.betweenness_centrality(
            self.graph,
            normalized=normalized,
            endpoints=False, 
#             weight_fn=lambda edge: 1.0 / edge if edge > 0 else 0 # Distance = 1/Similarity
        )
        
        return bc_scores

    def shortest_path_to_all(self, source_node):
        """
        Calculates Dijkstra's shortest path distances from source to all other nodes.
        Useful for analyzing connectivity spread.
        """
        # Computes shortest path lengths in parallel
        # Note: For similarity graphs, "distance" is usually inverse of similarity.
        distances = rx.dijkstra_shortest_path_lengths(
            self.graph,
            source_node,
            edge_cost_fn=lambda edge: 1.0 / edge if edge > 0 else float('inf')
        )
        return distances

In [165]:
sppmi_coo = sppmi.tocoo()

# m = sppmi_coo.data >= 4.5
m = sppmi_coo.data >= 5

# sppmi_sparse = csr_array((sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)
sppmi_sparse = csr_array((2**sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)

normalizer = sppmi_sparse.sum(axis=1)[:, None]
normalizer[normalizer > 0] = 1/normalizer[normalizer > 0]

sppmi_sparse = sppmi_sparse * normalizer

sppmi_sparse = sppmi_sparse.tocsr()

sppmi_sparse.sum(axis=1)

array([1., 1., 0., ..., 0., 0., 0.], shape=(295860,))

In [166]:
sppmi_sparse.nnz / sppmi.nnz

0.021356694223511948

In [175]:
analyzer = GraphAnalyzer(sppmi_sparse)

✅ Graph initialized with 295860 nodes and 5219070 edges.


In [168]:
target_node = a
print(f"\n--- Running RWR from Node {target_node} ---")
start = time.time()
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.4)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.5)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=1)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.1)


rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.8)

# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=.15)
# rwr_ranks = analyzer.random_walk_with_restart(target_node, restart_prob=0)

print(f"Done in {time.time() - start:.4f}s")
# print("Top 5 most related nodes:", rwr_ranks[:5])

top_k, _ = zip(*rwr_ranks)

top_k = list(top_k)

top_k.index(b)


--- Running RWR from Node 9060 ---
Done in 1.8013s


1

In [ ]:
bc = analyzer.betweenness_centrality(k=100)

In [76]:
scores = sppmi[a].toarray()
top_k = np.argsort(-scores).tolist()

top_k.index(b)

12

In [169]:
# top_k = top_k[:20]

In [170]:
top_k_matches = [idx2cat[idx] for idx in top_k]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'The Acid (spotify:artist:0bRtSoJSpQdnbB3dWrWprR)',
 'Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'Big Scary (spotify:artist:4mLYW48jy9Pwv6KpT74Evf)',
 'Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)',
 'Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)',
 'Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)',
 'Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)',
 'Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)',
 'The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)',
 'Matt Corby (spotify:artist:7CIW23FQUXPc1zebnO1TDG)',
 'Night Beds (spotify:artist:533wKOfkJylNSi6ntO1wXd)',
 'SOHN (spotify:arti

In [171]:
sppmi[a, b], sppmi[b, c]

(np.float64(5.15147708284573), np.float64(5.970563799185629))

In [30]:
(sppmi >= 5).sum()

np.int64(5219070)

In [29]:
sppmi.nnz / (sppmi >= 5).sum()

np.float64(46.82372606613822)

In [31]:
sppmi_coo = sppmi.tocoo()

In [32]:
sppmi_coo.row

array([     0,      0,      0, ..., 295859, 295859, 295859],
      shape=(244376304,), dtype=int32)

In [35]:
m = sppmi_coo.data >= 5
sppmi_sparse = csr_array((sppmi_coo.data[m], (sppmi_coo.row[m], sppmi_coo.col[m])), shape=sppmi.shape)

sppmi_sparse

In [42]:
top_k = 100

scores = sppmi[b].toarray()
scores[np.argsort(-scores)][:top_k]

array([10.40859166,  5.9705638 ,  5.92283519,  5.75857571,  5.75417824,
        5.61457528,  5.58217614,  5.36321438,  5.27224304,  5.21260147,
        5.15147708,  5.13813142,  5.12376079,  5.07039063,  5.05707438,
        5.04604413,  5.01342481,  5.01133322,  5.00189371,  4.98003152,
        4.93685378,  4.92941766,  4.92931417,  4.92275872,  4.89887195,
        4.89817005,  4.88836995,  4.88481719,  4.88274651,  4.87599625,
        4.86551547,  4.85763243,  4.82310904,  4.81807206,  4.80875945,
        4.79157969,  4.78271779,  4.78002297,  4.75489629,  4.75232499,
        4.73954468,  4.73421814,  4.72559807,  4.71709601,  4.71435745,
        4.70607899,  4.70255034,  4.69924759,  4.69512501,  4.69293094,
        4.68268953,  4.67197963,  4.67090346,  4.66093319,  4.65682033,
        4.65408017,  4.64520893,  4.64149286,  4.61258096,  4.61189032,
        4.61114063,  4.60847171,  4.60529802,  4.59793757,  4.58756201,
        4.58300921,  4.57978145,  4.5760959 ,  4.57569146,  4.57

In [5]:
import numpy as np
from scipy import sparse

def sparse_pmi(pxy, px, row_indices, col_indices):
    denominator = px[row_indices] * px[col_indices]
    
    pmi = pxy.copy()
    pmi_values = np.log2(pxy.data / denominator)
    pmi.data = pmi_values
    
    return pmi

In [13]:
def get_metric(i, j, sppmi):
    return np.argsort(-sppmi[i].toarray()).tolist().index(j)

In [6]:
pxy = X / n_users

In [7]:
px = X.diagonal() / n_users

In [8]:
X_coo = X.tocoo()

row_indices = X_coo.row
col_indices = X_coo.col

In [9]:
pmi = sparse_pmi(pxy, px, row_indices, col_indices)

In [23]:
def normalize_pmi(pxy, px, row_indices, col_indices, pmi, norm_type="npmi"):
    row_norm = -np.log2(px[row_indices])
    col_norm = -np.log2(px[col_indices])
    
    if norm_type == "npmi":
        norm = -np.log2(pxy.data)
    elif norm_type == "mean":
        norm = (row_norm + col_norm)/2
    elif norm_type == "sum":
        norm = row_norm + col_norm
    elif norm_type == "min":
        norm = np.where(row_norm < col_norm, row_norm, col_norm)
    elif norm_type == "max":
        norm = np.where(row_norm > col_norm, row_norm, col_norm)
    
    pmi = pmi.copy()
    pmi.data /= norm
    
    return pmi

In [24]:
pmi2 = normalize_pmi(pxy, px, row_indices, col_indices, pmi, "sum")

metrics = [
    get_metric(a, b, pmi2),
    get_metric(b, a, pmi2),
    
#     get_metric(b, c, pmi2),
#     get_metric(c, b, pmi2),
]

np.mean(metrics)

np.float64(1070.5)

In [49]:
pxy[a, b] / (px[a] * px[b])

np.float64(39.20742196497797)

In [52]:
np.log2(pxy[a, b] / (px[a] * px[b]))

np.float64(5.293054877251469)

In [59]:
np.log2(pxy[c, c] / (px[c] * px[c])), -np.log2(pxy[c, c]), -np.log2(px[c])

(np.float64(13.781821449819493),
 np.float64(13.781821449819493),
 np.float64(13.781821449819493))

In [60]:
np.log2(pxy[b, c] / (px[b] * px[c])) / -np.log2(px[c])

np.float64(0.6107501137038661)

In [61]:
np.log2(pxy[b, c] / (px[b] * px[c])) / -np.log2(px[b])

np.float64(0.748528064568391)

In [63]:
np.log2(pxy[b, c] / (px[b] * px[c])) / max(-np.log2(px[c]), -np.log2(px[b]))

np.float64(0.6107501137038661)

In [55]:
np.log2(pxy[b, c] / (px[b] * px[c])) / np.log2(pxy[c, c] / (px[c] * px[c]))

np.float64(0.6107501137038661)

In [54]:
np.log2(pxy[b, c] / (px[b] * px[c])) / np.log2(pxy[b, b] / (px[b] * px[b]))

np.float64(0.748528064568391)

In [51]:
# pxy[a, c] / (px[a] * px[c])

In [47]:
pxy[b, c] / (px[b] * px[c])

np.float64(341.8569670449884)

In [48]:
pxy[c, b] / (px[c] * px[b])

np.float64(341.8569670449884)